# Dock 01

## Configuration

In [121]:
ligdictjson = "preprocessed/ligand.json"
prodictjson = "preprocessed/protein.json"

prodir = "preprocessed/proteinprep/"
dockeddir = "preprocessed/ligandprep/"
idealdir = "preprocessed/Ideal_Ligand/"
resultsdir = "results/"
docked = resultsdir + "docked.sdf"
log = resultsdir + "gninalog.txt"
rmsdlog = resultsdir + "rmsd.txt"
masterlog = resultsdir + "masterlog.txt"
results = resultsdir + "results.csv"

In [2]:
sourceproteinprocesseddir = "../proteinprep01/chemfiles/"
sourceligandidealdir = "../mfaber_workflow/ER_Ligand_Prep/Ideal_Ligand/"
localprocesseddir = "preprocessed/"

In [3]:
experimentdescription = "Dock 01"

In [4]:
proteindict = {}
dockedliganddict = {}
idealliganddict = {}
gninaoptdict = {}
prepdict = {}

In [5]:
import copy
def reportdict(rows, columns):
    lines = []
    if len(rows) == 0:
        return (lines) 
    inner_keys = set()
    for r in rows.values():
        inner_keys.update(r.keys())
    col_widths = {}
    col_widths["id"] = max(len("id"), max(len(k) for k in rows))
    for col in inner_keys:
        header_len = len(col)
        data_len = max(len(str(r.get(col, ""))) for r in rows.values())
        col_widths[col] = max(header_len, data_len)
    header = "  ".join(f"{col:<{col_widths[col]}}" for col in columns)
    lines.append(header)
    for outer_key, inner in rows.items():
        cells = [f"{outer_key:<{col_widths['id']}}"]
        for col in columns[1:]:
            cells.append(f"{str(inner.get(col, '')):<{col_widths[col]}}")
        lines.append("  ".join(cells))
    return (lines)

In [6]:
#copy preprocessed proteins and docked ligand files
!cp -R {sourceproteinprocesseddir}* {localprocesseddir}
!cp -R {sourceligandidealdir} {localprocesseddir}
!ls -al {localprocesseddir}

total 32
drwxr-xr-x 6 dwaine dwaine 4096 Mar 19 11:08 .
drwxr-xr-x 5 dwaine dwaine 4096 Mar 19 11:22 ..
drwxr-xr-x 2 dwaine dwaine 4096 Mar 19 11:08 Ideal_Ligand
-rw-r--r-- 1 dwaine dwaine 2953 Mar 19 11:23 ligand.json
drwxr-xr-x 2 dwaine dwaine 4096 Mar 19 10:49 ligandprep
-rw-r--r-- 1 dwaine dwaine 3850 Mar 19 11:23 protein.json
drwxr-xr-x 2 dwaine dwaine 4096 Mar 19 10:49 proteindownload
drwxr-xr-x 2 dwaine dwaine 4096 Mar 19 10:49 proteinprep


## Populate docked ligand dictionary

In [9]:
dockedliganddict = {}

In [10]:
import json

# Read from text file
with open(ligdictjson, "r") as f:
    dockedliganddict = json.load(f)

#Which ligands are available?
lines = reportdict(dockedliganddict, ["id","source","prep","localfilename"])
print("\n".join(lines))

id               source    prep          localfilename        
EST_redock_1ERE  1ERE.pdb  rdkit_save_H  EST_redock_1ERE_A.sdf
EST_redock_1GWR  1GWR.pdb  rdkit_save_H  EST_redock_1GWR_A.sdf
EST_redock_3UUD  3UUD.pdb  rdkit_save_H  EST_redock_3UUD_A.sdf
EST_redock_6CBZ  6CBZ.pdb  rdkit_save_H  EST_redock_6CBZ_A.sdf
DES_redock_3ERD  3ERD.pdb  rdkit_save_H  DES_redock_3ERD_A.sdf
DES_redock_4ZN7  4ZN7.pdb  rdkit_save_H  DES_redock_4ZN7_A.sdf
27M_redock_4MGC  4MGC.pdb  rdkit_save_H  27M_redock_4MGC_A.sdf
27J_redock_4MG8  4MG8.pdb  rdkit_save_H  27J_redock_4MG8_A.sdf
36J_redock_4TUZ  4TUZ.pdb  rdkit_save_H  36J_redock_4TUZ_A.sdf
2OH_redock_3UU7  3UU7.pdb  rdkit_save_H  2OH_redock_3UU7_A.sdf
27K_redock_4MG9  4MG9.pdb  rdkit_save_H  27K_redock_4MG9_A.sdf
27L_redock_4MGA  4MGA.pdb  rdkit_save_H  27L_redock_4MGA_A.sdf
EST_redock_1G50  1G50.pdb  rdkit_save_H  EST_redock_1G50_A.sdf


## Populate protein dictionary

In [8]:
import json

# Read from text file
with open(prodictjson, "r") as f:
    proteindict = json.load(f)    

lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed"])
print("\n".join(lines))

id    name  prep          localfilename_fixed
1ERE        PDBfixfunc74  1ERE_A_fixed.pdb   
1GWR        PDBfixfunc74  1GWR_A_fixed.pdb   
3UUD        PDBfixfunc74  3UUD_A_fixed.pdb   
6CBZ        PDBfixfunc74  6CBZ_A_fixed.pdb   
3ERD        PDBfixfunc74  3ERD_A_fixed.pdb   
4ZN7        PDBfixfunc74  4ZN7_A_fixed.pdb   
4MGC        PDBfixfunc74  4MGC_A_fixed.pdb   
4MG8        PDBfixfunc74  4MG8_A_fixed.pdb   
4TUZ        PDBfixfunc74  4TUZ_A_fixed.pdb   
3UU7        PDBfixfunc74  3UU7_A_fixed.pdb   
4MG9        PDBfixfunc74  4MG9_A_fixed.pdb   
4MGA        PDBfixfunc74  4MGA_A_fixed.pdb   
1G50        PDBfixfunc74  1G50_A_fixed.pdb   


In [71]:
# Add docked ligand id into protein dictionary. Next iteration this will happen in the 'pre process protein' code.

for (k1, v1), (k2, v2) in zip(proteindict.items(), dockedliganddict.items()):
    #print(f"Key: {k1}, Dict1: {v1}, Dict2: {v2}")
    v1['dockedid'] = v2['id']

lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed","dockedid"])
print("\n".join(lines))

id    name  prep          localfilename_fixed  dockedid       
1ERE        PDBfixfunc74  1ERE_A_fixed.pdb     EST_redock_1ERE
1GWR        PDBfixfunc74  1GWR_A_fixed.pdb     EST_redock_1GWR
3UUD        PDBfixfunc74  3UUD_A_fixed.pdb     EST_redock_3UUD
6CBZ        PDBfixfunc74  6CBZ_A_fixed.pdb     EST_redock_6CBZ
3ERD        PDBfixfunc74  3ERD_A_fixed.pdb     DES_redock_3ERD
4ZN7        PDBfixfunc74  4ZN7_A_fixed.pdb     DES_redock_4ZN7
4MGC        PDBfixfunc74  4MGC_A_fixed.pdb     27M_redock_4MGC
4MG8        PDBfixfunc74  4MG8_A_fixed.pdb     27J_redock_4MG8
4TUZ        PDBfixfunc74  4TUZ_A_fixed.pdb     36J_redock_4TUZ
3UU7        PDBfixfunc74  3UU7_A_fixed.pdb     2OH_redock_3UU7
4MG9        PDBfixfunc74  4MG9_A_fixed.pdb     27K_redock_4MG9
4MGA        PDBfixfunc74  4MGA_A_fixed.pdb     27L_redock_4MGA
1G50        PDBfixfunc74  1G50_A_fixed.pdb     EST_redock_1G50


## Populate ideal ligand dictionary

In [12]:
idealliganddict = {}

In [16]:
idealliganddict["27J"] = {'id':"27J", 'prep': "obabel-mmff94", 'localfilename': "27J.sdf"}
idealliganddict["27K"] = {'id':"27K", 'prep': "obabel-mmff94", 'localfilename': "27K.sdf"}
idealliganddict["27L"] = {'id':"27L", 'prep': "obabel-mmff94", 'localfilename': "27L.sdf"}

idealliganddict["27M"] = {'id':"27M", 'prep': "obabel-mmff94", 'localfilename': "27M.sdf"}
idealliganddict["2OH"] = {'id':"2OH", 'prep': "obabel-mmff94", 'localfilename': "2OH.sdf"}
idealliganddict["36J"] = {'id':"36J", 'prep': "obabel-mmff94", 'localfilename': "36J.sdf"}

idealliganddict["Caffeine"] = {'id':"Caffeine", 'prep': "obabel-mmff94", 'localfilename': "Caffeine.sdf"}
idealliganddict["DES"] = {'id':"DES", 'prep': "obabel-mmff94", 'localfilename': "DES.sdf"}
idealliganddict["DE2"] = {'id':"DE2", 'prep': "obabel-mmff94", 'localfilename': "DE2.sdf"}

idealliganddict["EST"] = {'id':"EST", 'prep': "obabel-mmff94", 'localfilename': "EST.sdf"}
idealliganddict["Melatonin"] = {'id':"Melatonin", 'prep': "obabel-mmff94", 'localfilename': "Melatonin.sdf"}
idealliganddict["Testosterone"] = {'id':"Testosterone", 'prep': "obabel-mmff94", 'localfilename': "Testosterone.sdf"}

In [17]:
lines = reportdict(idealliganddict, ["id","prep","localfilename"])
print("\n".join(lines))

id            prep           localfilename   
27J           obabel-mmff94  27J.sdf         
27K           obabel-mmff94  27K.sdf         
27L           obabel-mmff94  27L.sdf         
27M           obabel-mmff94  27M.sdf         
2OH           obabel-mmff94  2OH.sdf         
36J           obabel-mmff94  36J.sdf         
Caffeine      obabel-mmff94  Caffeine.sdf    
DES           obabel-mmff94  DES.sdf         
DE2           obabel-mmff94  DE2.sdf         
EST           obabel-mmff94  EST.sdf         
Melatonin     obabel-mmff94  Melatonin.sdf   
Testosterone  obabel-mmff94  Testosterone.sdf


## Examine proteins to determine what chains and ligands are present in the proteins. (Optional informational step) ##

In [21]:
import gemmi

for key, value in proteindict.items():
    
    structure = gemmi.read_structure(prodir + value['localfilename_fixed'])
    #structure = gemmi.read_structure(chemfilesdir + "6O4w_rcbs.pdb")
    ligands = []
    
    for model in structure:
        for chain in model:
            for res in chain:
                if res.het_flag != ' ':  # hetero-residue
                    if res.name not in ("HOH", "WAT", "H2O"):
                        #if res.seqid.num == 604:
                        ligands.append((res.name, chain.name, res.seqid.num))

    print("protein: " + value['localfilename'])
    print(set(ligands))

protein: 1ERE.pdb
{('LYS', 'A', 302), ('ALA', 'A', 361), ('LEU', 'A', 469), ('HIS', 'A', 513), ('PRO', 'A', 552), ('GLU', 'A', 380), ('MET', 'A', 343), ('GLU', 'A', 444), ('THR', 'A', 485), ('HIS', 'A', 476), ('ARG', 'A', 515), ('ILE', 'A', 326), ('GLN', 'A', 441), ('GLU', 'A', 471), ('MET', 'A', 315), ('ASP', 'A', 313), ('PHE', 'A', 425), ('TYR', 'A', 526), ('VAL', 'A', 533), ('LEU', 'A', 508), ('ASN', 'A', 455), ('LEU', 'A', 453), ('ASN', 'A', 519), ('THR', 'A', 460), ('LEU', 'A', 462), ('VAL', 'A', 316), ('SER', 'A', 341), ('ASN', 'A', 348), ('ALA', 'A', 491), ('PRO', 'A', 325), ('HIS', 'A', 524), ('MET', 'A', 528), ('THR', 'A', 496), ('LEU', 'A', 428), ('ASP', 'A', 480), ('GLN', 'A', 498), ('MET', 'A', 427), ('ASN', 'A', 439), ('GLY', 'A', 494), ('LEU', 'A', 327), ('THR', 'A', 334), ('TRP', 'A', 383), ('LEU', 'A', 391), ('TYR', 'A', 537), ('VAL', 'A', 355), ('LYS', 'A', 416), ('VAL', 'A', 364), ('HIS', 'A', 398), ('ILE', 'A', 358), ('ALA', 'A', 493), ('ASN', 'A', 359), ('VAL', 'A',

In [23]:
from Bio.PDB import PDBParser

for key, value in proteindict.items():
    
  parser = PDBParser(QUIET=True)
  structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
  #structure = parser.get_structure("prot", prodir + value['localfilename_fixed'])
    
  print("Protein: " + value['localfilename'])
    
  for model in structure:
    print(f"  Model {model.id}:")
    chain_ids = [chain.id for chain in model]
    print("    Chains:", ", ".join(chain_ids))


Protein: 1ERE.pdb
  Model 0:
    Chains: A
Protein: 1GWR.pdb
  Model 0:
    Chains: A
Protein: 3UUD.pdb
  Model 0:
    Chains: A
Protein: 6CBZ.pdb
  Model 0:
    Chains: A
Protein: 3ERD.pdb
  Model 0:
    Chains: A
Protein: 4ZN7.pdb
  Model 0:
    Chains: A
Protein: 4MGC.pdb
  Model 0:
    Chains: A
Protein: 4MG8.pdb
  Model 0:
    Chains: A
Protein: 4TUZ.pdb
  Model 0:
    Chains: A
Protein: 3UU7.pdb
  Model 0:
    Chains: A
Protein: 4MG9.pdb
  Model 0:
    Chains: A
Protein: 4MGA.pdb
  Model 0:
    Chains: A
Protein: 1G50.pdb
  Model 0:
    Chains: A


## Visualize Ligands

In [ ]:
from rdkit import Chem

ligandfile = ligdir + '2R6_ideal_PubChem.sdf'
ligandfileout = ligdir + '2R6_ideal_PubChem_NOH.sdf'

# Load SDF file (remove Hs on read - most efficient)
mol = Chem.MolFromMolFile(ligandfile, removeHs=True)

# Or if already loaded with Hs:
# mol = Chem.MolFromMolFile("ligand.sdf", removeHs=False)
# mol = Chem.RemoveHs(mol)

# Write H-free SDF
writer = Chem.SDWriter(ligandfileout)
writer.write(mol)
writer.close()

print(f"Atoms before: {Chem.MolFromMolFile(ligandfileout, removeHs=False).GetNumAtoms()}")
print(f"Atoms after:  {mol.GetNumAtoms()}")

In [ ]:
import nglview as nv
from rdkit import Chem

# From SDF
view = nv.show_structure_file(ligdir + '2R6_ideal_PubChem.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '4o09_final_ligand_2R6_A.pdb')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view = nv.show_structure_file(ligdir + '2R6_redock_4o09_final_A_obabel.sdf')
view.add_representation('ball+stick')
view.camera = 'orthographic'
view.center()
view

In [ ]:
view.display(gui=True) 

## GNINA and reporting functions

In [ ]:
#Examples
#output = !~/gnina-binary/gnina -r proteins_pdbfixtest/2ama_fixer_andy_minusH.pdb -l proteins_pdbfixtest/Trenbolone.sdf --autobox_ligand proteins_pdbfixtest/2ama-ligand.pdb -o docked.sdf.gz  --exhaustiveness=64 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --log proteins_pdbfixtest/log.txt
# Predict against complexed ligand.
#!~/octoberproject/gnina -r 8GUTciffixed4.pdb -l 8GUT-ligand.sdf --autobox_ligand 8GUT-ligand.sdf -o docked.sdf.gz  --exhaustiveness=64 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --log log.txt --no_gpu  

In [24]:
gninaoptdict = {}

gninaopt = {}
gninaopt["id"] = "gninaopt2"
gninaopt["opt"] = "--exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu" 
gninaoptdict["gninaopt2"] = gninaopt

In [25]:
print (gninaoptdict)
cols = ["id","opt"]
print(reportdict(gninaoptdict,cols))

{'gninaopt2': {'id': 'gninaopt2', 'opt': '--exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu'}}
['id         opt                                                                           ', 'gninaopt2  --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu']


In [26]:
# Call Gnina, parse results from RMSD calculation, call function that parse results from both log files and appends to results variable

#pro = subdir + "6O4w_rcbs_chainB_fixed.pdb"
#lig = subdir + "6O4w_rcbs_chainB_E20.sdf"
#box = subdir + "6O4w_rcbs_chainB_E20.sdf"
#docked = subdir + "docked.sdf"
#log = subdir + "gninalog.txt"
#!~/octoberproject/gnina -r "{pro}" -l "{lig}" --autobox_ligand "{box}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  

def rungnina(proteinid,ligandid,boxid):
    protein = proteindict[proteinid]
    ligand = idealliganddict[ligandid]
    box = dockedliganddict[boxid]
    p = prodir + protein["localfilename_fixed"]
    l = idealdir + ligand["localfilename"]
    b = dockeddir + box["localfilename"]
    #!~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=8 --seed 0 --pose_sort_order CNNaffinity --no_gpu  
    !~/octoberproject/gnina -r "{p}" -l "{l}" --autobox_ligand "{b}" -o "{docked}" --log "{log}" --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu  

    #Compute RMSD of two docked ligand files and save results to rmsdlog file.
    !obrms -f "{b}" "{docked}" | tee "{rmsdlog}"

    appendresults(proteinid,ligandid,boxid)

In [132]:
# Get values from text files, append to results data, print to screen.
# Note: The CNN_VS score is also report in the docked.sdf file.
import pandas as pd

def appendresults(pro, lig, box):
    max_rows = 1
    
    resultsdf = getdockedresults(docked)
    #for index, row in resultsdf.iterrows():
    for index, row in resultsdf.head(max_rows).iterrows():
        new_row = (pro + " + " + lig + " @ " + box , "{:.4f}".format(row['CNNscore']), "{:.4f}".format(row['CNN_VS']), "{:.4f}".format(row['RMSD']),lig,pro,box)
        data.append(new_row)

In [28]:
def getdockedresults(docked):
    from rdkit import Chem
    import pandas as pd  

    rmsddf = pd.read_csv(rmsdlog, sep=" ", header=None)
    
    rows = []
    for i, mol in enumerate(Chem.SDMolSupplier(docked)):
        if mol is None:
            continue
        rows.append({
            "pose": i,
            "CNNscore": float(mol.GetProp("CNNscore")),
            "CNN_VS": float(mol.GetProp("CNN_VS")),
            "CNNaffinity": float(mol.GetProp("CNNaffinity")),
            "RMSD": float(rmsddf.iloc[i,2]),
        })
    df = pd.DataFrame(rows)
    return(df)

In [60]:
from itertools import islice

def reporttable(rows):
    max_rows_to_report = None #first row is header labels. setting to '2' results in one row of results data.
    lines = []
    # rows: list of tuples, all same length
    if not rows:
        return
    # Number of columns
    n_cols = len(rows[0])
    # Max width per column (over all rows)
    widths = [
        max(len(str(row[col])) for row in rows)
        for col in range(n_cols)
    ]
    # Print each row with aligned columns
    #for row in rows:
    #for item in islice(data, max_iter):
    for row in islice(rows, max_rows_to_report):
        line = "  ".join(
            f"{str(value):<{widths[i]}}"
            for i, value in enumerate(row)
        )
        #print(line)
        lines.append(line)

    return (lines)

In [133]:
#Report out results with details of Ligands and Proteins used
from datetime import datetime

now = datetime.now()
stamp = now.strftime("%Y-%m-%d %H:%M:%S")
unixtime = int(datetime.now().timestamp())

def reportall():
    lines = []

    lines.append("Report produced " + stamp + " experiment id: " + str(unixtime))

    lines.append("")
    lines.append(experimentdescription)

    lines.append("")
    cols = ["name","functionname","description"]
    lines.append("File preparation descriptions:")
    lines = lines + reportdict(prepdict,cols)
                
    lines.append("")
    cols = ["id", "name","prep","localfilename_fixed"]
    lines.append("Proteins:")
    lines = lines + reportdict(proteindict, cols)

    lines.append("")
    cols = ["id","prep","localfilename"]
    lines.append("Ideal ligands:")
    lines = lines + reportdict(idealliganddict, cols)

    lines.append("")
    cols = ["id", "opt"]
    lines.append("Gnina options:")
    lines = lines + reportdict(gninaoptdict,cols)

    lines.append("")
    lines.append("Results:")
    newrow = ("proteinid + ligandid @ ligandid", "CNN Score", "CNN_VS", "RMSD", "ligand_ideal", "protein", "ligand_native")     
    outputlist = list(data)
    outputlist.insert(0, newrow)
    lines = lines + reporttable(outputlist)

    return lines;

## Call Gnina

In [72]:
#Which proteins are available?
lines = reportdict(proteindict, ["id","name","prep","localfilename_fixed","dockedid"])
print("\n".join(lines))

id    name  prep          localfilename_fixed  dockedid       
1ERE        PDBfixfunc74  1ERE_A_fixed.pdb     EST_redock_1ERE
1GWR        PDBfixfunc74  1GWR_A_fixed.pdb     EST_redock_1GWR
3UUD        PDBfixfunc74  3UUD_A_fixed.pdb     EST_redock_3UUD
6CBZ        PDBfixfunc74  6CBZ_A_fixed.pdb     EST_redock_6CBZ
3ERD        PDBfixfunc74  3ERD_A_fixed.pdb     DES_redock_3ERD
4ZN7        PDBfixfunc74  4ZN7_A_fixed.pdb     DES_redock_4ZN7
4MGC        PDBfixfunc74  4MGC_A_fixed.pdb     27M_redock_4MGC
4MG8        PDBfixfunc74  4MG8_A_fixed.pdb     27J_redock_4MG8
4TUZ        PDBfixfunc74  4TUZ_A_fixed.pdb     36J_redock_4TUZ
3UU7        PDBfixfunc74  3UU7_A_fixed.pdb     2OH_redock_3UU7
4MG9        PDBfixfunc74  4MG9_A_fixed.pdb     27K_redock_4MG9
4MGA        PDBfixfunc74  4MGA_A_fixed.pdb     27L_redock_4MGA
1G50        PDBfixfunc74  1G50_A_fixed.pdb     EST_redock_1G50


In [33]:
#Which ligands are available?
lines = reportdict(idealliganddict, ["id","prep","localfilename"])
print("\n".join(lines))

id            prep           localfilename   
27J           obabel-mmff94  27J.sdf         
27K           obabel-mmff94  27K.sdf         
27L           obabel-mmff94  27L.sdf         
27M           obabel-mmff94  27M.sdf         
2OH           obabel-mmff94  2OH.sdf         
36J           obabel-mmff94  36J.sdf         
Caffeine      obabel-mmff94  Caffeine.sdf    
DES           obabel-mmff94  DES.sdf         
DE2           obabel-mmff94  DE2.sdf         
EST           obabel-mmff94  EST.sdf         
Melatonin     obabel-mmff94  Melatonin.sdf   
Testosterone  obabel-mmff94  Testosterone.sdf


In [53]:
#Which docked ligands are available?
lines = reportdict(dockedliganddict, ["id","prep","localfilename"])
print("\n".join(lines))

id               prep          localfilename        
EST_redock_1ERE  rdkit_save_H  EST_redock_1ERE_A.sdf
EST_redock_1GWR  rdkit_save_H  EST_redock_1GWR_A.sdf
EST_redock_3UUD  rdkit_save_H  EST_redock_3UUD_A.sdf
EST_redock_6CBZ  rdkit_save_H  EST_redock_6CBZ_A.sdf
DES_redock_3ERD  rdkit_save_H  DES_redock_3ERD_A.sdf
DES_redock_4ZN7  rdkit_save_H  DES_redock_4ZN7_A.sdf
27M_redock_4MGC  rdkit_save_H  27M_redock_4MGC_A.sdf
27J_redock_4MG8  rdkit_save_H  27J_redock_4MG8_A.sdf
36J_redock_4TUZ  rdkit_save_H  36J_redock_4TUZ_A.sdf
2OH_redock_3UU7  rdkit_save_H  2OH_redock_3UU7_A.sdf
27K_redock_4MG9  rdkit_save_H  27K_redock_4MG9_A.sdf
27L_redock_4MGA  rdkit_save_H  27L_redock_4MGA_A.sdf
EST_redock_1G50  rdkit_save_H  EST_redock_1G50_A.sdf


In [163]:
data = []

In [164]:
#row = ["1ERE", "", "", ""]
#data.append(row)
#rungnina("1ERE", "EST", "EST_redock_1ERE")
#rungnina("1GWR", "EST", "EST_redock_1GWR")

from itertools import islice

pro = dict(islice(proteindict.items(), 2))
ide = dict(islice(idealliganddict.items(), 2))

for pkey, pvalue in proteindict.items():
#for pkey, pvalue in pro.items():
    for ikey, ivalue in idealliganddict.items():
    #for ikey, ivalue in ide.items():
        print(f"{pkey} vs {ikey} at {pvalue['dockedid']}")
        rungnina(pkey, ikey, pvalue['dockedid'])

1ERE vs 27J at EST_redock_1ERE
              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/Ideal_Ligand/27J.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
2999413 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mo

[13:53:16] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:53:16] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:53:16] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:53:16] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:53:16] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:53:16] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:53:16] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:53:16] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:53:16] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/Ideal_Ligand/27K.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
7184 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | aff

[13:54:54] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:54:54] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:54:54] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:54:54] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:54:54] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:54:54] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:54:54] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:54:54] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[13:54:54] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1ERE_A_fixed.pdb -l preprocessed/Ideal_Ligand/27L.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1ERE_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
8814 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score | aff

...

[16:14:40] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:14:40] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:14:40] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:14:40] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:14:40] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:14:40] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:14:40] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:14:40] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:14:40] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1G50_A_fixed.pdb -l preprocessed/Ideal_Ligand/Melatonin.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1G50_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
896 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose score 

[16:16:00] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:00] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:00] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:00] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:00] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:00] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:00] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:00] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:00] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

              _             
             (_)            
   __ _ _ __  _ _ __   __ _ 
  / _` | '_ \| | '_ \ / _` |
 | (_| | | | | | | | | (_| |
  \__, |_| |_|_|_| |_|\__,_|
   __/ |                    
  |___/                     

gnina v1.3.2 master:f23dd2b   Built Jul 29 2025.
gnina is based on smina and AutoDock Vina.
Please cite appropriately.

Commandline: /home/dwaine/octoberproject/gnina -r preprocessed/proteinprep/1G50_A_fixed.pdb -l preprocessed/Ideal_Ligand/Testosterone.sdf --autobox_ligand preprocessed/ligandprep/EST_redock_1G50_A.sdf -o results/docked.sdf --log results/gninalog.txt --exhaustiveness=16 --num_modes=9 --seed 0 --pose_sort_order CNNscore --no_gpu
Using random seed: 0

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************
6013 | pose 0 | initial pose not within box

mode |  affinity  |  intramol  |    CNN     |   CNN
     | (kcal/mol) | (kcal/mol) | pose sc

[16:16:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol as 3D.
[16:16:35] Warning: molecule is tagged as 2D, but at least one Z coordinate is not zero. Marking the mol

## Display all results

In [165]:
print(data)


[('1ERE + 27J @ EST_redock_1ERE', '0.6477', '4.6075', 'inf', '27J', '1ERE', 'EST_redock_1ERE'), ('1ERE + 27K @ EST_redock_1ERE', '0.8632', '4.9930', 'inf', '27K', '1ERE', 'EST_redock_1ERE'), ('1ERE + 27L @ EST_redock_1ERE', '0.9241', '6.5886', 'inf', '27L', '1ERE', 'EST_redock_1ERE'), ('1ERE + 27M @ EST_redock_1ERE', '0.7627', '4.3862', 'inf', '27M', '1ERE', 'EST_redock_1ERE'), ('1ERE + 2OH @ EST_redock_1ERE', '0.8773', '6.2580', 'inf', '2OH', '1ERE', 'EST_redock_1ERE'), ('1ERE + 36J @ EST_redock_1ERE', '0.7878', '5.3286', 'inf', '36J', '1ERE', 'EST_redock_1ERE'), ('1ERE + Caffeine @ EST_redock_1ERE', '0.5773', '3.0018', 'inf', 'Caffeine', '1ERE', 'EST_redock_1ERE'), ('1ERE + DES @ EST_redock_1ERE', '0.9153', '7.4409', 'inf', 'DES', '1ERE', 'EST_redock_1ERE'), ('1ERE + DE2 @ EST_redock_1ERE', '0.9153', '7.4409', 'inf', 'DE2', '1ERE', 'EST_redock_1ERE'), ('1ERE + EST @ EST_redock_1ERE', '0.9799', '8.1912', '0.5071', 'EST', '1ERE', 'EST_redock_1ERE'), ('1ERE + Melatonin @ EST_redock_1ERE

In [141]:
#This block presents the results to stdout.
lines = reportall()
text = "\n".join(lines) + "\n"
print(text)

Report produced 2026-03-23 13:13:06 experiment id: 1774296786

Dock 01

File preparation descriptions:

Proteins:
id    name  prep          localfilename_fixed
1ERE        PDBfixfunc74  1ERE_A_fixed.pdb   
1GWR        PDBfixfunc74  1GWR_A_fixed.pdb   
3UUD        PDBfixfunc74  3UUD_A_fixed.pdb   
6CBZ        PDBfixfunc74  6CBZ_A_fixed.pdb   
3ERD        PDBfixfunc74  3ERD_A_fixed.pdb   
4ZN7        PDBfixfunc74  4ZN7_A_fixed.pdb   
4MGC        PDBfixfunc74  4MGC_A_fixed.pdb   
4MG8        PDBfixfunc74  4MG8_A_fixed.pdb   
4TUZ        PDBfixfunc74  4TUZ_A_fixed.pdb   
3UU7        PDBfixfunc74  3UU7_A_fixed.pdb   
4MG9        PDBfixfunc74  4MG9_A_fixed.pdb   
4MGA        PDBfixfunc74  4MGA_A_fixed.pdb   
1G50        PDBfixfunc74  1G50_A_fixed.pdb   

Ideal ligands:
id            prep           localfilename   
27J           obabel-mmff94  27J.sdf         
27K           obabel-mmff94  27K.sdf         
27L           obabel-mmff94  27L.sdf         
27M           obabel-mmff94  27M.sdf      

In [ ]:
#This block will append results listed above to master results file.

with open(masterlog, "a") as f:
    f.write(text)

## Notes

In [159]:
print (data)

[('1ERE + 27J @ EST_redock_1ERE', '0.6477', '4.6075', 'inf', '27J', '1ERE', 'EST_redock_1ERE'), ('1ERE + 27K @ EST_redock_1ERE', '0.8632', '4.9930', 'inf', '27K', '1ERE', 'EST_redock_1ERE'), ('1GWR + 27J @ EST_redock_1GWR', '0.7177', '4.9734', 'inf', '27J', '1GWR', 'EST_redock_1GWR'), ('1GWR + 27K @ EST_redock_1GWR', '0.9231', '5.1058', 'inf', '27K', '1GWR', 'EST_redock_1GWR')]


In [166]:
import pandas as pd

df3 = pd.DataFrame(data)
df3.columns = ["protein + ideal + docked", "CNN_pose", "CNN_VS", "RMSD", "ideal_ligand", "protein", "native_ligand"]
#df3["ligand"] = ["27J","27K","27J","27K"]
print(df3)

                  protein + ideal + docked CNN_pose  CNN_VS    RMSD  \
0             1ERE + 27J @ EST_redock_1ERE   0.6477  4.6075     inf   
1             1ERE + 27K @ EST_redock_1ERE   0.8632  4.9930     inf   
2             1ERE + 27L @ EST_redock_1ERE   0.9241  6.5886     inf   
3             1ERE + 27M @ EST_redock_1ERE   0.7627  4.3862     inf   
4             1ERE + 2OH @ EST_redock_1ERE   0.8773  6.2580     inf   
..                                     ...      ...     ...     ...   
151           1G50 + DES @ EST_redock_1G50   0.8925  7.3899     inf   
152           1G50 + DE2 @ EST_redock_1G50   0.8925  7.3899     inf   
153           1G50 + EST @ EST_redock_1G50   0.9686  8.1152  0.8367   
154     1G50 + Melatonin @ EST_redock_1G50   0.3722  2.3354     inf   
155  1G50 + Testosterone @ EST_redock_1G50   0.8560  6.8655  0.8710   

     ideal_ligand protein    native_ligand  
0             27J    1ERE  EST_redock_1ERE  
1             27K    1ERE  EST_redock_1ERE  
2           

## create and write .csv

In [167]:
import csv

# Write one line (creates file if it doesn't exist, overwrites if it does)
cols = ["ideal_ligand", "protein", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"]
with open(results, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(cols)

In [168]:
for group_name, group_df in df3.groupby("ideal_ligand"):
    #print("Group:", group_name)

    df_sorted = group_df.sort_values(by="CNN_pose", ascending=False)

    cols = ["ideal_ligand", "protein", "native_ligand", "CNN_pose", "CNN_VS", "RMSD"]  # desired order
    #print to stdout
    print(df_sorted[cols])
    #append to .csv file
    df_sorted[cols].to_csv(results, mode="a", index=False, header=False)
    

    ideal_ligand protein    native_ligand CNN_pose  CNN_VS    RMSD
36           27J    6CBZ  EST_redock_6CBZ   0.7356  4.8308     inf
12           27J    1GWR  EST_redock_1GWR   0.7177  4.9734     inf
96           27J    4TUZ  36J_redock_4TUZ   0.7070  4.5855  1.4256
108          27J    3UU7  2OH_redock_3UU7   0.7014  5.0870     inf
24           27J    3UUD  EST_redock_3UUD   0.6836  4.6035     inf
84           27J    4MG8  27J_redock_4MG8   0.6757  4.4561  1.4254
48           27J    3ERD  DES_redock_3ERD   0.6752  4.3226     inf
0            27J    1ERE  EST_redock_1ERE   0.6477  4.6075     inf
120          27J    4MG9  27K_redock_4MG9   0.6374  4.1574  2.0097
60           27J    4ZN7  DES_redock_4ZN7   0.6327  4.1277     inf
72           27J    4MGC  27M_redock_4MGC   0.5956  3.7478     inf
132          27J    4MGA  27L_redock_4MGA   0.4519  2.8396     inf
144          27J    1G50  EST_redock_1G50   0.3929  2.5507     inf
    ideal_ligand protein    native_ligand CNN_pose  CNN_VS    